In [26]:
import os
import sys
sys.path.append('/Users/bytedance/dev/Compiler')
import argparse
from datetime import datetime

from transformers import (
    AutoModelForMaskedLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    GPT2Tokenizer, GPT2Model, GPT2Config,
    BertTokenizer, BertModel, BertConfig,
    EncoderDecoderModel, EncoderDecoderConfig
)
from datasets import load_from_disk
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.config import load_config
from src.utils.utils import get_logger
from src.model import Inst2VecTokenizer

In [28]:

class GPT2DecoderWithAutophase(GPT2Model):
    def __init__(self, config):
        super().__init__(config)
        
        # 1. 定义静态特征投影层
        self.autophase_input_dim = 16
        self.autophase_proj = nn.Linear(
            in_features=self.autophase_input_dim,
            out_features=config.n_embd  # GPT-2中隐藏层维度为n_embd（对应BERT的hidden_size）
        )
        
        # 2. 定义dropout层（与GPT-2配置对齐，防止过拟合）
        self.autophase_dropout = nn.Dropout(config.embd_pdrop)
    
    def forward(
        self,
        hidden_states,  # GPT-2解码器嵌入层输出的词嵌入特征 (batch_size, dec_seq_len, n_embd)
        autophase=None,  # 新增：静态特征输入 (batch_size, static_feature_input_dim)
        attention_mask=None,
        encoder_hidden_states=None,  # 编码器输出的上下文特征（交叉注意力层使用）
        encoder_attention_mask=None,
        past_key_values=None,
        use_cache=None,
        output_attentions=False,
        output_hidden_states=False,
        return_dict=True,
    ):
        # 3. 静态特征融合逻辑（嵌入层后、第一个Transformer层之前，与BERT解码器逻辑一致）
        if autophase is not None:
            autophase_proj = self.autophase_proj(autophase)
            autophase_proj = self.autophase_dropout(autophase_proj)
            # c. 广播匹配解码序列长度：(batch_size, n_embd) -> (batch_size, dec_seq_len, n_embd)
            # 确保每个解码token都能获得静态特征的约束（GPT自回归生成时，每一步seq_len=1，逻辑依然有效）
            dec_seq_len = hidden_states.shape[1]
            autophase_proj_broadcast = autophase_proj.unsqueeze(1).repeat(1, dec_seq_len, 1)
            
            # d. 特征融合（优先选择「逐元素相加」，稳定高效，适配GPT的特征分布）
            hidden_states = hidden_states + autophase_proj_broadcast

        # 4. 调用父类forward方法，继续GPT-2解码器后续的计算（自注意力、交叉注意力等）
        return super().forward(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

class Passformer(EncoderDecoderModel):
    def __init__(self, config=None, encoder=None, decoder=None):
        super().__init__(config, encoder, decoder)
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        autophase=None,  # 新增：传递给GPT-2解码器的静态特征
        encoder_outputs=None,
        past_key_values=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        # 统一配置默认参数（与原生模型对齐）
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        # 关键修正1：从self.config.decoder中获取use_cache
        use_cache = use_cache if use_cache is not None else self.config.decoder.use_cache
        # 关键修正2：从self.config.decoder中获取output_attentions
        output_attentions = output_attentions if output_attentions is not None else self.config.decoder.output_attentions
        # 关键修正3：从self.config.decoder中获取output_hidden_states
        output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.decoder.output_hidden_states

        
        # 第一步：编码器前向传播（BERT编码器，生成上下文特征，与原生逻辑一致）
        if encoder_outputs is None:
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )
        elif return_dict and not isinstance(encoder_outputs, BaseModelOutput):
            encoder_outputs = BaseModelOutput(
                last_hidden_state=encoder_outputs[0],
                hidden_states=encoder_outputs[1] if len(encoder_outputs) > 1 else None,
                attentions=encoder_outputs[2] if len(encoder_outputs) > 2 else None,
            )
        
        # 第二步：GPT-2解码器输入处理（提取解码器词嵌入）
        if decoder_input_ids is not None:
            decoder_embeds = self.decoder.embeddings(input_ids=decoder_input_ids)
        else:
            decoder_embeds = None
        
        # 第三步：自定义GPT-2解码器前向传播（传递静态特征）
        decoder_outputs = self.decoder(
            hidden_states=decoder_embeds,  # GPT-2词嵌入作为输入
            static_features=autophase,  # 传入静态特征
            attention_mask=decoder_attention_mask,
            encoder_hidden_states=encoder_outputs.last_hidden_state,  # 接收BERT编码器输出
            encoder_attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        
        # 第四步：处理输出（与原生Seq2Seq模型对齐，生成最终logits）
        if not return_dict:
            return decoder_outputs + encoder_outputs
        
        return Seq2SeqModelOutput(
            logits=decoder_outputs.logits if hasattr(decoder_outputs, 'logits') else None,
            past_key_values=decoder_outputs.past_key_values,
            decoder_hidden_states=decoder_outputs.hidden_states,
            decoder_attentions=decoder_outputs.attentions,
            cross_attentions=decoder_outputs.cross_attentions,
            encoder_last_hidden_state=encoder_outputs.last_hidden_state,
            encoder_hidden_states=encoder_outputs.hidden_states,
            encoder_attentions=encoder_outputs.attentions,
        )

In [30]:
bert_encoder = BertModel(BertConfig())
gpt2_decoder = GPT2DecoderWithAutophase(GPT2Config())
model = Passformer(encoder=bert_encoder, decoder=gpt2_decoder)
# 示例输入
input_ids = torch.randint(0, 1000, (2, 5))
decoder_input_ids = torch.randint(0, 1000, (2, 5))
autophase = torch.randn(2, 16)  # 静态特征

# 模型调用
outputs = model(
    input_ids=input_ids,
    decoder_input_ids=decoder_input_ids,
    autophase=autophase
)

print("Logits shape:", outputs.logits.shape)

AttributeError: 'GPT2DecoderWithAutophase' object has no attribute 'embeddings'